# India Flight Delay — Data Analysis & Feature Engineering

## 1. Project Overview

This notebook covers the data preparation and exploratory analysis stage of the India Flight Delay Analytics & Prediction System.

The objective is to understand the structure and quality of the flight dataset, identify patterns associated with delays, and prepare a clean analytical dataset for SQL, Power BI, Tableau, and machine learning.

### Key Objectives

- Inspect the dataset structure and data types
- Identify missing values and duplicate records
- Analyze the distribution of flight delays
- Compare delay patterns across airlines, airports, routes, weather conditions, and operational factors
- Engineer business-friendly features and categories
- Validate important findings using descriptive analysis
- Export the final dataset for downstream analytics and machine learning

## 2. Data Quality Assessment

The initial dataset contains **150,000 flight records** and **23 columns**.

### Data Quality Results

- Records: **150,000**
- Columns: **23**
- Missing values: **None**
- Duplicate records: **None**

The dataset was therefore suitable for further exploratory analysis without requiring missing-value imputation or duplicate removal.

### Target Distribution

The `Delay_Target` variable contains:

- **129,925 delayed flights**
- **20,075 on-time flights**

This represents an overall observed delay rate of approximately **86.62%** in the dataset.

Because the target classes are imbalanced, stratified sampling was used during machine learning model development.

## 3. Feature Engineering

Additional features were created to make the dataset easier to analyze and interpret from a business perspective.

### Engineered Features

- `Scheduled_Departure_Minutes` — scheduled departure time converted into minutes after midnight
- `Departure_Time_Band` — groups flights into Early Morning, Morning, Afternoon, and Evening
- `Congestion_Band` — categorizes airport congestion into Low, Medium, High, and Very High
- `Turnaround_Risk_Band` — categorizes turnaround risk into Low, Medium, High, and Very High
- `Previous_Delay_Band` — groups previous flight delay into meaningful operational ranges
- `Route` — combines origin and destination airports into a single route identifier

These features were created to support easier segmentation and visualization in SQL, Power BI, Tableau, and machine learning analysis.

## 4. Exploratory Data Analysis — Key Findings

### Overall Delay Performance

The dataset contains 150,000 flights, of which 129,925 were classified as delayed.

The overall observed delay rate is **86.62%**, with an average departure delay of approximately **42.66 minutes**.

### Airline Analysis

Observed delay rates across airlines were relatively close, ranging from approximately **85.42% to 87.83%**.

This indicates that airline-level differences were relatively small compared with some operational variables in this dataset.

### Airport Analysis

Among airports with at least 500 flights, observed delay rates varied across origin airports.

Cochin (COK) recorded the highest observed delay rate among the major airports analyzed, at approximately **93.31%**.

### Congestion Analysis

Airport congestion showed a strong relationship with observed delay rates:

| Congestion Level | Observed Delay Rate |
|---|---:|
| Low | 5.07% |
| Medium | 36.99% |
| High | 94.69% |
| Very High | 95.89% |

The large increase across congestion levels makes airport congestion an important operational variable for further investigation.

### Turnaround Risk

Turnaround risk was also strongly associated with observed delays:

| Turnaround Risk | Observed Delay Rate |
|---|---:|
| Low | 39.66% |
| Medium | 92.22% |
| High | 95.90% |

The Very High category contained only 13 flights and was therefore considered too small for meaningful interpretation.

### Previous Flight Delay

Flights with greater previous-flight delays generally showed higher observed delay rates.

This suggests that disruption from an earlier flight may carry forward into subsequent operations.

### Weather

Weather categories showed some variation in observed delay rates:

- Clear/Partly Cloudy: **86.18%**
- Rain: **88.07%**
- Heavy Rain: **91.62%**

The Cloudy category contained only four flights and was excluded from business interpretation because of its extremely small sample size.

### Important Interpretation Note

These findings describe **observed relationships within the dataset**. They should not be interpreted as evidence that a particular factor directly causes flight delays.

The dataset is used for portfolio analysis and predictive modeling, so real-world operational decisions would require validation against production data.

## 5. Final Dataset Export

The cleaned and feature-engineered dataset was exported as:

`india_flight_delay_final.csv`

The final dataset contains **150,000 records and 29 columns** and is used as the common analytical dataset for:

- PostgreSQL SQL analysis
- Power BI dashboard development
- Machine learning prediction
- Future Tableau analysis

The exported dataset preserves the original analytical variables while adding business-friendly features such as route, departure time band, congestion band, turnaround risk band, and previous delay band.

---

## 6. Conclusion

The exploratory analysis identified several important patterns in flight delay performance.

Airport congestion and turnaround risk showed particularly strong relationships with observed delays, while previous-flight delay also provided useful operational information.

These findings were subsequently carried forward into the machine learning stage, where a predictive model was developed to estimate the probability of flight delay.

The project therefore combines descriptive analytics, database analysis, business intelligence, and predictive analytics into a single end-to-end flight delay decision-support solution.

Load the data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("india_flight_delay_model_input.csv")

df.head()

,Flight_Date,Airline,Flight_Number,Origin_Airport,Destination_Airport,Scheduled_Departure_Hour,Scheduled_Departure_Minute,Day_of_Week,Month,Is_Weekend,...,Humidity_pct,Wind_Speed_kmh,Visibility_km,Rainfall_mm,Cloud_Cover_pct,Origin_Congestion_Index,Previous_Flight_Delay_Minutes,Turnaround_Risk_Index,Departure_Delay,Delay_Target
0,2025-12-04,IndiGo,IN8723,BOM,DEL,13,57,3,12,0,...,46.1,4.3,9.8,5.2,0.0,63.5,0,23.9,43,1
1,2024-01-15,IndiGo,IN558,CCU,HYD,19,14,0,1,0,...,70.5,8.9,11.7,0.4,39.8,95.3,1,31.5,49,1
2,2023-11-06,Vistara,VI4949,HYD,DEL,8,50,0,11,0,...,53.7,7.8,9.6,4.3,16.6,100.0,1,41.5,73,1
3,2025-08-08,SpiceJet,SP3196,BLR,DIB,11,56,4,8,0,...,42.0,1.5,12.6,4.6,23.6,51.1,1,28.2,40,1
4,2023-09-09,IndiGo,IN8458,BLR,JGA,22,59,5,9,1,...,58.5,1.3,11.3,3.5,36.2,67.0,0,27.9,0,0


In [ ]:
df.shape

(150000, 23)

Understand the columns

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 23 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Flight_Date                    150000 non-null  object 
 1   Airline                        150000 non-null  object 
 2   Flight_Number                  150000 non-null  object 
 3   Origin_Airport                 150000 non-null  object 
 4   Destination_Airport            150000 non-null  object 
 5   Scheduled_Departure_Hour       150000 non-null  int64  
 6   Scheduled_Departure_Minute     150000 non-null  int64  
 7   Day_of_Week                    150000 non-null  int64  
 8   Month                          150000 non-null  int64  
 9   Is_Weekend                     150000 non-null  int64  
 10  Peak_Hour                      150000 non-null  int64  
 11  Weather                        150000 non-null  object 
 12  Temperature_C                 

In [ ]:
df.describe(include="all")

,Flight_Date,Airline,Flight_Number,Origin_Airport,Destination_Airport,Scheduled_Departure_Hour,Scheduled_Departure_Minute,Day_of_Week,Month,Is_Weekend,...,Humidity_pct,Wind_Speed_kmh,Visibility_km,Rainfall_mm,Cloud_Cover_pct,Origin_Congestion_Index,Previous_Flight_Delay_Minutes,Turnaround_Risk_Index,Departure_Delay,Delay_Target
count,150000,150000,150000,150000,150000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,...,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000
unique,1095,9,50854,86,86,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2024-12-29,IndiGo,IN1650,DEL,DEL,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,180,60242,17,26337,23642,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,14.001260,29.533773,2.997107,6.512973,0.286820,...,56.332991,5.993765,10.891581,1.666205,25.013692,72.406212,6.170807,35.715187,42.664340,0.866167
std,NaN,NaN,NaN,NaN,NaN,5.480291,17.322034,2.003977,3.445167,0.452278,...,10.161597,4.223185,1.271557,2.409718,13.020552,19.277706,6.483483,8.719541,19.016712,0.340474
min,NaN,NaN,NaN,NaN,NaN,5.000000,0.000000,0.000000,1.000000,0.000000,...,20.000000,0.000000,5.100000,0.000000,0.000000,0.000000,0.000000,0.400000,0.000000,0.000000
25%,NaN,NaN,NaN,NaN,NaN,9.000000,15.000000,1.000000,4.000000,0.000000,...,49.500000,2.900000,10.000000,0.000000,15.800000,60.900000,1.000000,29.900000,39.000000,1.000000
50%,NaN,NaN,NaN,NaN,NaN,14.000000,30.000000,3.000000,7.000000,0.000000,...,56.300000,5.000000,10.900000,0.600000,24.800000,72.700000,4.000000,35.700000,47.000000,1.000000
75%,NaN,NaN,NaN,NaN,NaN,19.000000,45.000000,5.000000,10.000000,1.000000,...,63.200000,8.100000,11.700000,2.500000,33.800000,88.800000,9.000000,41.500000,54.000000,1.000000


In [ ]:
df.columns

Index(['Flight_Date', 'Airline', 'Flight_Number', 'Origin_Airport',
       'Destination_Airport', 'Scheduled_Departure_Hour',
       'Scheduled_Departure_Minute', 'Day_of_Week', 'Month', 'Is_Weekend',
       'Peak_Hour', 'Weather', 'Temperature_C', 'Humidity_pct',
       'Wind_Speed_kmh', 'Visibility_km', 'Rainfall_mm', 'Cloud_Cover_pct',
       'Origin_Congestion_Index', 'Previous_Flight_Delay_Minutes',
       'Turnaround_Risk_Index', 'Departure_Delay', 'Delay_Target'],
      dtype='object')

Check missing values

In [ ]:
df.isnull().sum()

,0
Flight_Date,0
Airline,0
Flight_Number,0
Origin_Airport,0
Destination_Airport,0
Scheduled_Departure_Hour,0
Scheduled_Departure_Minute,0
Day_of_Week,0
Month,0
Is_Weekend,0


In [ ]:
df.isnull().mean() * 100

,0
Flight_Date,0.0
Airline,0.0
Flight_Number,0.0
Origin_Airport,0.0
Destination_Airport,0.0
Scheduled_Departure_Hour,0.0
Scheduled_Departure_Minute,0.0
Day_of_Week,0.0
Month,0.0
Is_Weekend,0.0


Check duplicates

In [ ]:
df.duplicated().sum()

np.int64(0)

Check the target

In [ ]:
df["Delay_Target"].value_counts()

,count
Delay_Target,
1,129925
0,20075


In [ ]:
df["Delay_Target"].value_counts(normalize=True) * 100

,proportion
Delay_Target,
1,86.616667
0,13.383333


# **Target & Delay Analysis**

In [ ]:
# Check the relationship between actual delay and target
pd.crosstab(
    pd.cut(
        df["Departure_Delay"],
        bins=[-1, 0, 15, 30, 60, 120, float("inf")]
    ),
    df["Delay_Target"]
)

Delay_Target,0,1
Departure_Delay,,
"(-1.0, 0.0]",20075,0
"(15.0, 30.0]",0,1116
"(30.0, 60.0]",0,113052
"(60.0, 120.0]",0,15757


In [ ]:
df.groupby("Delay_Target")["Departure_Delay"].describe()

,count,mean,std,min,25%,50%,75%,max
Delay_Target,,,,,,,,
0,20075.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
1,129925.0,49.256502,9.633457,19.0,42.0,48.0,55.0,119.0


In [ ]:
df[["Departure_Delay", "Delay_Target"]].corr()

,Departure_Delay,Delay_Target
Departure_Delay,1.000000,0.881886
Delay_Target,0.881886,1.000000


In [ ]:
df.groupby("Delay_Target")[
    ["Previous_Flight_Delay_Minutes",
     "Turnaround_Risk_Index",
     "Origin_Congestion_Index",
     "Rainfall_mm",
     "Visibility_km"]
].mean()

,Previous_Flight_Delay_Minutes,Turnaround_Risk_Index,Origin_Congestion_Index,Rainfall_mm,Visibility_km
Delay_Target,,,,,
0,4.271880,26.628672,48.149001,1.478790,10.917644
1,6.464214,37.119165,76.154247,1.695163,10.887554


Understand the categorical variables

In [ ]:
df["Airline"].value_counts()

,count
Airline,
IndiGo,60242
Air India,26875
Air India Express,15212
Akasa Air,11937
SpiceJet,11739
Vistara,8994
Star Air,6072
Alliance Air,5915
Fly91,3014


In [ ]:
df["Weather"].value_counts()

,count
Weather,
Clear/Partly Cloudy,119250
Rain,28801
Heavy Rain,1945
Cloudy,4


In [ ]:
df["Origin_Airport"].nunique(), df["Destination_Airport"].nunique()

(86, 86)

In [ ]:
df["Origin_Airport"].value_counts().head(10)

,count
Origin_Airport,
DEL,26337
BOM,20363
BLR,17398
HYD,14461
MAA,13019
CCU,11697
COK,10184
AMD,8579
PBD,393


In [ ]:
pd.crosstab(
    pd.cut(
        df["Departure_Delay"],
        bins=[-1, 0, 15, 30, 60, 120, float("inf")],
        include_lowest=True
    ),
    df["Delay_Target"]
)

Delay_Target,0,1
Departure_Delay,,
"(-1.001, 0.0]",20075,0
"(15.0, 30.0]",0,1116
"(30.0, 60.0]",0,113052
"(60.0, 120.0]",0,15757


In [ ]:
df.groupby("Delay_Target")[
    [
        "Previous_Flight_Delay_Minutes",
        "Turnaround_Risk_Index",
        "Origin_Congestion_Index",
        "Rainfall_mm",
        "Visibility_km",
        "Temperature_C",
        "Humidity_pct",
        "Wind_Speed_kmh",
        "Cloud_Cover_pct"
    ]
].mean()

,Previous_Flight_Delay_Minutes,Turnaround_Risk_Index,Origin_Congestion_Index,Rainfall_mm,Visibility_km,Temperature_C,Humidity_pct,Wind_Speed_kmh,Cloud_Cover_pct
Delay_Target,,,,,,,,,
0,4.271880,26.628672,48.149001,1.478790,10.917644,27.023821,56.252394,5.94396,24.809768
1,6.464214,37.119165,76.154247,1.695163,10.887554,27.028704,56.345444,6.00146,25.045201


In [ ]:
df.groupby("Airline")["Delay_Target"].mean().sort_values(ascending=False)

,Delay_Target
Airline,
Alliance Air,0.878276
SpiceJet,0.875799
Air India Express,0.874967
Star Air,0.873847
Fly91,0.869940
Air India,0.868837
IndiGo,0.862305
Vistara,0.858128
Akasa Air,0.854151


# Does airport congestion increase the probability of delay?

In [ ]:
df["Congestion_Band"] = pd.cut(
    df["Origin_Congestion_Index"],
    bins=[0, 25, 50, 75, 100],
    labels=["Low", "Medium", "High", "Very High"]
)

df.groupby("Congestion_Band")["Delay_Target"].agg(
    ["count", "mean"]
)

/tmp/ipykernel_2676/3104170247.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Congestion_Band")["Delay_Target"].agg(


,count,mean
Congestion_Band,,
Low,1914,0.050679
Medium,19399,0.369864
High,61397,0.946854
Very High,67284,0.958906


Does turnaround risk affect delays?

In [ ]:
df["Turnaround_Risk_Band"] = pd.cut(
    df["Turnaround_Risk_Index"],
    bins=[0, 25, 50, 75, 100],
    labels=["Low", "Medium", "High", "Very High"]
)

df.groupby("Turnaround_Risk_Band")["Delay_Target"].agg(
    ["count", "mean"]
)

/tmp/ipykernel_2676/4043743087.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Turnaround_Risk_Band")["Delay_Target"].agg(


,count,mean
Turnaround_Risk_Band,,
Low,16494,0.396568
Medium,126127,0.922150
High,7366,0.959001
Very High,13,0.923077


Does a previous flight delay increase the chance of the next flight being delayed?

In [ ]:
df["Previous_Delay_Band"] = pd.cut(
    df["Previous_Flight_Delay_Minutes"],
    bins=[-1, 0, 15, 30, 60, float("inf")],
    labels=["None", "1-15 min", "16-30 min", "31-60 min", "60+ min"]
)

df.groupby("Previous_Delay_Band")["Delay_Target"].agg(
    ["count", "mean"]
)

/tmp/ipykernel_2676/1946585736.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Previous_Delay_Band")["Delay_Target"].agg(


,count,mean
Previous_Delay_Band,,
None,30375,0.818469
1-15 min,105989,0.869977
16-30 min,12659,0.941781
31-60 min,974,0.956879
60+ min,3,0.666667


In [ ]:
df["Departure_Delay"].value_counts().sort_index().head(30)

,count
Departure_Delay,
0,20075
19,1
20,1
21,3
22,6
23,7
24,14
25,22
26,61


# Weather analysis 🌧️

In [ ]:
df.groupby("Weather")["Delay_Target"].agg(
    ["count", "mean"]
).sort_values("mean", ascending=False)

,count,mean
Weather,,
Cloudy,4,1.000000
Heavy Rain,1945,0.916195
Rain,28801,0.880733
Clear/Partly Cloudy,119250,0.861828


In [ ]:
df.groupby("Peak_Hour")["Delay_Target"].agg(
    ["count", "mean"]
)

,count,mean
Peak_Hour,,
0,78708,0.792257
1,71292,0.947764


In [ ]:
df.groupby("Is_Weekend")["Delay_Target"].agg(
    ["count", "mean"]
)

,count,mean
Is_Weekend,,
0,106977,0.847930
1,43023,0.911512


In [ ]:
df.groupby("Scheduled_Departure_Hour")["Delay_Target"].agg(
    ["count", "mean"]
).sort_index()

,count,mean
Scheduled_Departure_Hour,,
5,7862,0.797253
6,7960,0.790704
7,7912,0.946158
8,7875,0.948063
9,7943,0.946368
10,7843,0.946067
11,7942,0.790229
12,7931,0.790821
13,7843,0.796889


In [ ]:
# Create a working copy
df_final = df.copy()

# Convert date to datetime
df_final["Flight_Date"] = pd.to_datetime(df_final["Flight_Date"])

# Create departure time in minutes
df_final["Scheduled_Departure_Minutes"] = (
    df_final["Scheduled_Departure_Hour"] * 60
    + df_final["Scheduled_Departure_Minute"]
)

# Create time-of-day category
df_final["Departure_Time_Band"] = pd.cut(
    df_final["Scheduled_Departure_Hour"],
    bins=[0, 6, 12, 18, 24],
    labels=["Early Morning", "Morning", "Afternoon", "Evening"],
    right=False
)

# Create congestion bands
df_final["Congestion_Band"] = pd.cut(
    df_final["Origin_Congestion_Index"],
    bins=[0, 25, 50, 75, 100],
    labels=["Low", "Medium", "High", "Very High"]
)

# Create turnaround risk bands
df_final["Turnaround_Risk_Band"] = pd.cut(
    df_final["Turnaround_Risk_Index"],
    bins=[0, 25, 50, 75, 100],
    labels=["Low", "Medium", "High", "Very High"]
)

# Create previous-flight delay bands
df_final["Previous_Delay_Band"] = pd.cut(
    df_final["Previous_Flight_Delay_Minutes"],
    bins=[-1, 0, 15, 30, 60, float("inf")],
    labels=["None", "1-15 min", "16-30 min", "31-60 min", "60+ min"]
)

# Create route
df_final["Route"] = (
    df_final["Origin_Airport"]
    + " → "
    + df_final["Destination_Airport"]
)

df_final.head()

,Flight_Date,Airline,Flight_Number,Origin_Airport,Destination_Airport,Scheduled_Departure_Hour,Scheduled_Departure_Minute,Day_of_Week,Month,Is_Weekend,...,Previous_Flight_Delay_Minutes,Turnaround_Risk_Index,Departure_Delay,Delay_Target,Congestion_Band,Turnaround_Risk_Band,Previous_Delay_Band,Scheduled_Departure_Minutes,Departure_Time_Band,Route
0,2025-12-04,IndiGo,IN8723,BOM,DEL,13,57,3,12,0,...,0,23.9,43,1,High,Low,None,837,Afternoon,BOM → DEL
1,2024-01-15,IndiGo,IN558,CCU,HYD,19,14,0,1,0,...,1,31.5,49,1,Very High,Medium,1-15 min,1154,Evening,CCU → HYD
2,2023-11-06,Vistara,VI4949,HYD,DEL,8,50,0,11,0,...,1,41.5,73,1,Very High,Medium,1-15 min,530,Morning,HYD → DEL
3,2025-08-08,SpiceJet,SP3196,BLR,DIB,11,56,4,8,0,...,1,28.2,40,1,High,Medium,1-15 min,716,Morning,BLR → DIB
4,2023-09-09,IndiGo,IN8458,BLR,JGA,22,59,5,9,1,...,0,27.9,0,0,High,Medium,None,1379,Evening,BLR → JGA


In [ ]:
df_final.to_csv("india_flight_delay_final.csv", index=False)

In [ ]:
from google.colab import files

files.download("india_flight_delay_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>